# Tarea 2 - Introducción a Data Science

Integrantes:
- Axel Mondaca
- Sebastián Hernández

In [325]:
import pandas as pd
from sklearn.model_selection import train_test_split    
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score # MSE y R2
from sklearn.preprocessing import StandardScaler, LabelEncoder
from prettytable import PrettyTable

In [326]:
df = pd.read_csv('housing_Tarea2.csv')
df

,house_type,house_size,location,city,latitude,longitude,price,currency,numBathrooms,numBalconies,isNegotiable,priceSqFt,verificationDate,description,SecurityDeposit,Status
0,1 RK Studio Apartment,400 sq ft,Kalkaji,Delhi,28.545561,77.254349,22000,INR,1.0,NaN,NaN,NaN,Posted a day ago,"Fully furnished, loaded with amenities & gadge...",No Deposit,Furnished
1,1 RK Studio Apartment,400 sq ft,Mansarover Garden,Delhi,28.643259,77.132828,20000,INR,1.0,NaN,NaN,NaN,Posted 9 days ago,Here is an excellent 1 BHK Independent Floor a...,No Deposit,Furnished
2,2 BHK Independent Floor,500 sq ft,Uttam Nagar,Delhi,28.618677,77.053352,8500,INR,1.0,NaN,NaN,NaN,Posted 12 days ago,"Zero Brokerage.\n\n2 Room set, Govt bijali Met...",No Deposit,Semi-Furnished
3,3 BHK Independent House,"1,020 sq ft",Model Town,Delhi,28.712898,77.180000,48000,INR,3.0,NaN,NaN,NaN,Posted a year ago,Itâs a 3 bhk independent house situated in M...,No Deposit,Furnished
4,2 BHK Apartment,810 sq ft,Sector 13 Rohini,Delhi,28.723539,77.131424,20000,INR,2.0,NaN,NaN,NaN,Posted a year ago,Well designed 2 bhk multistorey apartment is a...,No Deposit,Unfurnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4 BHK Villa,"5,896 sq ft",Sunder Nagar,Delhi,28.618437,76.961784,1022001,INR,4.0,2.0,NaN,NaN,Posted 2 months ago,Its four bhk villa in the super location of De...,"40,10,102",Unfurnished
4996,5 BHK Independent House,"6,521 sq ft",Sunder Nagar,Delhi,28.618437,76.961784,1549181,INR,4.0,2.0,NaN,NaN,Posted 2 months ago,A 5 bhk property is available for rent in Sund...,"54,01,015",Unfurnished
4997,3 BHK Independent Floor,"1,855 sq ft",New Friends Colony,Delhi,28.567051,77.273560,301012,INR,3.0,2.0,NaN,NaN,Posted 2 months ago,Its three bhk builder floor in the super locat...,"18,18,181",Unfurnished
4998,3 BHK Independent Floor,"2,856 sq ft",New Friends Colony,Delhi,28.567051,77.273560,301011,INR,3.0,2.0,NaN,NaN,Posted 2 months ago,Its three bhk builder floor in the super locat...,"10,10,110",Unfurnished


# Limpieza de Datos

In [328]:
# Limpieza de datos para la columna house_type
df["numRooms"] = df["house_type"].str[0].astype(int)
# 3 BHK <- 3 Bedroom, 1 Hall, 1 Kitchen
df["house_type"].value_counts()
df["rooms"] = df["house_type"].str.split(" ").str[1]
df["housing_type"] = df["house_type"].str.split(" ").str[2:].str.join(" ")

# Encoding características categóricas
encoder = LabelEncoder()
df["rooms"] = encoder.fit_transform(df["rooms"]).astype(int)
df["housing_type"] = encoder.fit_transform(df["housing_type"]).astype(int)
df.drop(columns=["house_type"], inplace=True)

In [329]:
# Limpieza columna house_size
df["house_size"] = df["house_size"].str.replace(",","") 
df["house_size"] = df["house_size"].str.split(" ").str[0].astype(int)

In [330]:
df["SecurityDeposit"] = df["SecurityDeposit"].str.replace("No Deposit","0") 
df["SecurityDeposit"] = df["SecurityDeposit"].str.replace(",","") 

Se reemplazan los valores de la columna "Security Deposit" por 0 en caso de que no haya y se elimina la coma en el valor

In [331]:
df["Status"].value_counts()
df["Status"] = df["Status"].str.replace("Unfurnished","0") 
df["Status"] = df["Status"].str.replace("Semi-Furnished","1") 

In [332]:
df["Status"] = df["Status"].str.replace("Furnished","2") 

Se reemplazan los valores de la columna "Status" por 0, 1 o 2, "Furnished" se reemplaza más abajo para evitar que se reemplace en caso de encontrarse con "Semi-Furnished"

In [333]:
df["isNegotiable"] = df["isNegotiable"].str.replace("Negotiable","1") 
df.fillna(0, inplace=True)

Se reemplaza los valores de la columna "isNegotiable" por "1" en caso de que sea "Negotiable" y si no tiene el dato se llena con valor 0.

In [334]:
df["location"] = encoder.fit_transform(df["location"]).astype(int)
df["city"] = encoder.fit_transform(df["city"]).astype(int)
df["currency"] = encoder.fit_transform(df["currency"]).astype(int)
df = df.fillna(0) # Rellenar valores nulos con 0


In [335]:
verif_map = {
    'day' : 1, 'days' : 1,
    "week" : 7, "weeks" : 7, 
    "month" : 30, "months" : 30,
    "year" : 365, "year" : 365
}
df[["cant_veces_str", "mult"]] = df["verificationDate"].str.extract(r'Posted\s+(\d+|a|an)\s+(\w+)')
df['cant_veces'] = df['cant_veces_str'].replace({'a': 1, 'an': 1}).astype(int)
df["days_since_verif"] = df["cant_veces"] * df["mult"].map(verif_map) # .apply()
df.drop(columns=["verificationDate", "cant_veces_str", "mult", "cant_veces"], inplace=True)
df

,house_size,location,city,latitude,longitude,price,currency,numBathrooms,numBalconies,isNegotiable,priceSqFt,description,SecurityDeposit,Status,numRooms,rooms,housing_type,days_since_verif
0,400,88,0,28.545561,77.254349,22000,0,1.0,0.0,0,0.0,"Fully furnished, loaded with amenities & gadge...",0,2,1,1,3,1.0
1,400,124,0,28.643259,77.132828,20000,0,1.0,0.0,0,0.0,Here is an excellent 1 BHK Independent Floor a...,0,2,1,1,3,9.0
2,500,259,0,28.618677,77.053352,8500,0,1.0,0.0,0,0.0,"Zero Brokerage.\n\n2 Room set, Govt bijali Met...",0,1,2,0,1,12.0
3,1020,133,0,28.712898,77.180000,48000,0,3.0,0.0,0,0.0,Itâs a 3 bhk independent house situated in M...,0,2,3,0,2,365.0
4,810,201,0,28.723539,77.131424,20000,0,2.0,0.0,0,0.0,Well designed 2 bhk multistorey apartment is a...,0,0,2,0,0,365.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,5896,249,0,28.618437,76.961784,1022001,0,4.0,2.0,0,0.0,Its four bhk villa in the super location of De...,4010102,0,4,0,4,60.0
4996,6521,249,0,28.618437,76.961784,1549181,0,4.0,2.0,0,0.0,A 5 bhk property is available for rent in Sund...,5401015,0,5,0,2,60.0
4997,1855,146,0,28.567051,77.273560,301012,0,3.0,2.0,0,0.0,Its three bhk builder floor in the super locat...,1818181,0,3,0,1,60.0
4998,2856,146,0,28.567051,77.273560,301011,0,3.0,2.0,0,0.0,Its three bhk builder floor in the super locat...,1010110,0,3,0,1,60.0


In [336]:
df.describe()

,house_size,location,city,latitude,longitude,price,currency,numBathrooms,numBalconies,priceSqFt,numRooms,rooms,housing_type,days_since_verif
count,5000.000000,5000.000000,5000.0,5000.000000,5000.000000,5.000000e+03,5000.0,5000.000000,5000.000000,5000.0,5000.00000,5000.00000,5000.000000,4315.000000
mean,2982.885400,136.314400,0.0,28.578012,77.174499,2.221738e+05,0.0,2.904000,1.069800,0.0,3.09780,0.02760,1.210400,147.619235
std,2168.663368,79.323699,0.0,0.190186,0.115636,2.739843e+05,0.0,1.104458,1.053731,0.0,1.14162,0.16384,1.045347,136.706730
min,150.000000,0.000000,0.0,20.011379,72.771332,3.000000e+03,0.0,0.000000,0.000000,0.0,1.00000,0.00000,0.000000,1.000000
25%,1100.000000,65.000000,0.0,28.544489,77.138248,2.950000e+04,0.0,2.000000,0.000000,0.0,2.00000,0.00000,1.000000,20.000000
50%,2500.000000,141.000000,0.0,28.569295,77.196472,1.250000e+05,0.0,3.000000,1.000000,0.0,3.00000,0.00000,1.000000,150.000000
75%,5896.000000,195.000000,0.0,28.618687,77.228950,3.011020e+05,0.0,4.000000,2.000000,0.0,4.00000,0.00000,1.000000,180.000000
max,14521.000000,287.000000,0.0,28.805466,80.361313,3.010101e+06,0.0,10.000000,8.000000,0.0,9.00000,1.00000,5.000000,365.000000


In [337]:
df.drop(columns=["description"], inplace=True)

In [338]:
df_baseline = df.copy()
df_baseline.describe()

,house_size,location,city,latitude,longitude,price,currency,numBathrooms,numBalconies,priceSqFt,numRooms,rooms,housing_type,days_since_verif
count,5000.000000,5000.000000,5000.0,5000.000000,5000.000000,5.000000e+03,5000.0,5000.000000,5000.000000,5000.0,5000.00000,5000.00000,5000.000000,4315.000000
mean,2982.885400,136.314400,0.0,28.578012,77.174499,2.221738e+05,0.0,2.904000,1.069800,0.0,3.09780,0.02760,1.210400,147.619235
std,2168.663368,79.323699,0.0,0.190186,0.115636,2.739843e+05,0.0,1.104458,1.053731,0.0,1.14162,0.16384,1.045347,136.706730
min,150.000000,0.000000,0.0,20.011379,72.771332,3.000000e+03,0.0,0.000000,0.000000,0.0,1.00000,0.00000,0.000000,1.000000
25%,1100.000000,65.000000,0.0,28.544489,77.138248,2.950000e+04,0.0,2.000000,0.000000,0.0,2.00000,0.00000,1.000000,20.000000
50%,2500.000000,141.000000,0.0,28.569295,77.196472,1.250000e+05,0.0,3.000000,1.000000,0.0,3.00000,0.00000,1.000000,150.000000
75%,5896.000000,195.000000,0.0,28.618687,77.228950,3.011020e+05,0.0,4.000000,2.000000,0.0,4.00000,0.00000,1.000000,180.000000
max,14521.000000,287.000000,0.0,28.805466,80.361313,3.010101e+06,0.0,10.000000,8.000000,0.0,9.00000,1.00000,5.000000,365.000000


In [339]:
df_baseline.fillna(0, inplace=True)

# Conjunto Limpio


In [340]:
clean_df = df_baseline.copy()
clean_df

,house_size,location,city,latitude,longitude,price,currency,numBathrooms,numBalconies,isNegotiable,priceSqFt,SecurityDeposit,Status,numRooms,rooms,housing_type,days_since_verif
0,400,88,0,28.545561,77.254349,22000,0,1.0,0.0,0,0.0,0,2,1,1,3,1.0
1,400,124,0,28.643259,77.132828,20000,0,1.0,0.0,0,0.0,0,2,1,1,3,9.0
2,500,259,0,28.618677,77.053352,8500,0,1.0,0.0,0,0.0,0,1,2,0,1,12.0
3,1020,133,0,28.712898,77.180000,48000,0,3.0,0.0,0,0.0,0,2,3,0,2,365.0
4,810,201,0,28.723539,77.131424,20000,0,2.0,0.0,0,0.0,0,0,2,0,0,365.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,5896,249,0,28.618437,76.961784,1022001,0,4.0,2.0,0,0.0,4010102,0,4,0,4,60.0
4996,6521,249,0,28.618437,76.961784,1549181,0,4.0,2.0,0,0.0,5401015,0,5,0,2,60.0
4997,1855,146,0,28.567051,77.273560,301012,0,3.0,2.0,0,0.0,1818181,0,3,0,1,60.0
4998,2856,146,0,28.567051,77.273560,301011,0,3.0,2.0,0,0.0,1010110,0,3,0,1,60.0


In [341]:
clean_df.describe()

,house_size,location,city,latitude,longitude,price,currency,numBathrooms,numBalconies,priceSqFt,numRooms,rooms,housing_type,days_since_verif
count,5000.000000,5000.000000,5000.0,5000.000000,5000.000000,5.000000e+03,5000.0,5000.000000,5000.000000,5000.0,5000.00000,5000.00000,5000.000000,5000.000000
mean,2982.885400,136.314400,0.0,28.578012,77.174499,2.221738e+05,0.0,2.904000,1.069800,0.0,3.09780,0.02760,1.210400,127.395400
std,2168.663368,79.323699,0.0,0.190186,0.115636,2.739843e+05,0.0,1.104458,1.053731,0.0,1.14162,0.16384,1.045347,136.765496
min,150.000000,0.000000,0.0,20.011379,72.771332,3.000000e+03,0.0,0.000000,0.000000,0.0,1.00000,0.00000,0.000000,0.000000
25%,1100.000000,65.000000,0.0,28.544489,77.138248,2.950000e+04,0.0,2.000000,0.000000,0.0,2.00000,0.00000,1.000000,14.000000
50%,2500.000000,141.000000,0.0,28.569295,77.196472,1.250000e+05,0.0,3.000000,1.000000,0.0,3.00000,0.00000,1.000000,60.000000
75%,5896.000000,195.000000,0.0,28.618687,77.228950,3.011020e+05,0.0,4.000000,2.000000,0.0,4.00000,0.00000,1.000000,180.000000
max,14521.000000,287.000000,0.0,28.805466,80.361313,3.010101e+06,0.0,10.000000,8.000000,0.0,9.00000,1.00000,5.000000,365.000000


In [342]:
clean_df.drop(columns=["priceSqFt","city","currency"], inplace=True)
clean_df.drop(columns=["isNegotiable"], inplace=True)
clean_df.drop(columns=["SecurityDeposit"], inplace=True)
clean_df.drop(columns=["Status"], inplace=True)

Eliminamos las columnas "PriceSqFt", "city" y "currency", ya que los datos que contienen no aportan información al modelo al ser todos los datos 0 en las últimas dos y tener un alto número de datos faltantes en comparación a la cantidad total para el primero. 

Las columnas "Status", "Security Deposit" y "isNegotiable" tienen un número bajo de variables, por lo que no aportan mucho al modelo.

In [343]:
scaler = StandardScaler()
clean_df["house_size"] = scaler.fit_transform(clean_df[["house_size"]])
clean_df["latitude"] = scaler.fit_transform(clean_df[["latitude"]]) 
clean_df["longitude"] = scaler.fit_transform(clean_df[["longitude"]])
clean_df["days_since_verif"] = scaler.fit_transform(clean_df[["days_since_verif"]])

Estandarizamos los datos de las columnas "house_size", "latitude", "longitude y "days since verif"

In [344]:
clean_df

,house_size,location,latitude,longitude,price,numBathrooms,numBalconies,numRooms,rooms,housing_type,days_since_verif
0,-1.191122,88,-0.170645,0.690597,22000,1.0,0.0,1,1,3,-0.924269
1,-1.191122,124,0.343105,-0.360396,20000,1.0,0.0,1,1,3,-0.865768
2,-1.145007,259,0.213840,-1.047752,8500,1.0,0.0,2,0,1,-0.843831
3,-0.905204,133,0.709306,0.047583,48000,3.0,0.0,3,0,2,1.737488
4,-1.002047,201,0.765263,-0.372537,20000,2.0,0.0,2,0,0,1.737488
...,...,...,...,...,...,...,...,...,...,...,...
4995,1.343411,249,0.212577,-1.839692,1022001,4.0,2.0,4,0,4,-0.492830
4996,1.631636,249,0.212577,-1.839692,1549181,4.0,2.0,5,0,2,-0.492830
4997,-0.520135,146,-0.057638,0.856745,301012,3.0,2.0,3,0,1,-0.492830
4998,-0.058514,146,-0.057638,0.856745,301011,3.0,2.0,3,0,1,-0.492830


In [345]:
clean_df.describe()

,house_size,location,latitude,longitude,price,numBathrooms,numBalconies,numRooms,rooms,housing_type,days_since_verif
count,5.000000e+03,5000.000000,5.000000e+03,5.000000e+03,5.000000e+03,5000.000000,5000.000000,5000.00000,5000.00000,5000.000000,5.000000e+03
mean,-9.094947e-17,136.314400,3.291944e-15,1.909939e-14,2.221738e+05,2.904000,1.069800,3.09780,0.02760,1.210400,6.821210e-17
std,1.000100e+00,79.323699,1.000100e+00,1.000100e+00,2.739843e+05,1.104458,1.053731,1.14162,0.16384,1.045347,1.000100e+00
min,-1.306412e+00,0.000000,-4.504799e+01,-3.808148e+01,3.000000e+03,0.000000,0.000000,1.00000,0.00000,0.000000,-9.315810e-01
25%,-8.683108e-01,65.000000,-1.762815e-01,-3.135147e-01,2.950000e+04,2.000000,0.000000,2.00000,0.00000,1.000000,-8.292058e-01
50%,-2.226873e-01,141.000000,-4.583800e-02,1.900424e-01,1.250000e+05,3.000000,1.000000,3.00000,0.00000,1.000000,-4.928300e-01
75%,1.343411e+00,195.000000,2.138906e-01,4.709363e-01,3.011020e+05,4.000000,2.000000,4.00000,0.00000,1.000000,3.846720e-01
max,5.320913e+00,287.000000,1.196076e+00,2.756166e+01,3.010101e+06,10.000000,8.000000,9.00000,1.00000,5.000000,1.737488e+00


In [346]:
df_with_interaction = clean_df.copy()
df_with_interaction["latitude_*_longitud"] = df_with_interaction["latitude"] * df_with_interaction["longitude"]


Creamos el conjunto con interacción y agregamos la columna "latitude_*_longitud" la cual es la multiplicación de los valores de las columnas "latitude" y "longitud"

# Regresión Lineal

In [347]:
# Variables
X = df_baseline.drop(columns=["price"])  # Variables independientes
y = df_baseline["price"]                 # Variable dependiente

# División del conjunto
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=254)
 

X2 = clean_df.drop(columns=["price"])  # Variables independientes
y2 = clean_df["price"]                 # Variable dependiente

# División del conjunto
X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, train_size=0.8, random_state=254)

X3 = df_with_interaction.drop(columns=["price"])  # Variables independientes
y3 = df_with_interaction["price"]                 # Variable dependiente

# División del conjunto
X_train3, X_test3, y_train3, y_test3 = train_test_split(X3, y3, train_size=0.8, random_state=254)

Primero dividimos los datos de prueba de cada conjunto en datos de entrenamiento y prueba en un 80% y 20% respectivamente

In [348]:
# Ajuste/Entrenamiento del modelo
lr = LinearRegression()
lr.fit(X_train, y_train)
y_prediction = lr.predict(X_test) # y_gorrito

lr2 = LinearRegression()
lr2.fit(X_train2, y_train2)
y_prediction2 = lr2.predict(X_test2) # y_gorrito

lr3 = LinearRegression()
lr3.fit(X_train3, y_train3)
y_prediction3 = lr3.predict(X_test3) # y_gorrito

Se crea un modelo de regresión lineal para cada conjunto

In [349]:
mse = mean_squared_error(y_test, y_prediction)
r2 = r2_score(y_test, y_prediction)
mse2 = mean_squared_error(y_test2, y_prediction2)
r22 = r2_score(y_test2, y_prediction2)
mse3 = mean_squared_error(y_test3, y_prediction3)
r23 = r2_score(y_test3, y_prediction3)

Se calcula el MSE y el valor de R2 para cada conjunto

# Tabla comparativa de los tres modelos

In [350]:
table = PrettyTable()
table.field_names = ["Modelo", "MSE", "R2"]
table.add_row(["Baseline", mse, r2])
table.add_row(["Limpio", mse2, r22])
table.add_row(["Limpio + interacción", mse3, r23])
table

Modelo,MSE,R2
Baseline,7537504863.499019,0.9115972764946418
Limpio,36960214057.050835,0.56651655379927
Limpio + interacción,36894077886.14375,0.5672922239628428


Según lo que podemos ver en la tabla, los modelos limpio y con interacción se ajustan mejor a los datos en comparación al modelo baseline, al tener un valor de MSE mucho menor y por ende más cercano a cero, la diferencia entre los modelos limpio y con interacción en cambio es bastante baja, con valores similares para tanto el MSE como el R2, sin embargo entre los dos se puede ver que el modelo con interacción es ligeramente mejor, con una diferencia de (7.75670164)^-4 en R2 y 66136170.91 para el MSE